# z711 — vuelta al z702, con las fugas corregidas

## Por qué este notebook existe

El leaderboard dice, con siete puntos, que cada "mejora" que le hice al z702 lo empeoró:

| pipeline | qué cambió | LB |
|---|---|---|
| **z702** | clase `tn(t+2)/s(t)`, clusters DTW por **par**, 15 features, validación por febreros | **0,245** |
| z705 | clase como **offset** `log(s)`, clusters por producto, 20 features | 0,258 |
| z709 | offset + peso `s^(p−1)`, 24 features, folds repartidos por todo el año | 0,314 |

Y el multiplicador perdió **7 de 7 veces**, en las dos familias y sin una sola excepción:

```
  z702   x0.9 0.252  |  x1.0 0.245  |  x1.1 0.268
  z705   m0.848 0.297  |  m0.970 0.262  |  m1.000 0.258  |  m1.092 0.260
```

Así que este notebook **no es una mejora del z709: es el z702**, con lo único que no admite
discusión —las fugas— corregido. Nada de modelado se cambia sin que el leaderboard lo pida.

## El juez interno: el fold 201902

La corrida de la v1 dio **WAPE 0,2708 en el ancla 201902** y el leaderboard devolvió **0,275**.
No es casualidad: ese fold es el análogo exacto del deploy — origen diciembre, target febrero,
horizonte 2. **Usá ese número para decidir y dejá de gastar submits.** El promedio de los cinco
folds (0,3030) es más pesimista porque incluye meses estructuralmente más difíciles: el peor es
201906, con origen abril, que sobre-predijo 20 % (39.073 contra 32.535 reales).

## Sobre "todo escalado" (z711 v1 dio 0,275)

La consigna es que todo vaya escalado, y se respeta: **las 18 features son relativas o
adimensionales**. Pero había un problema real detrás. Con el target dividido *y* las features
también divididas, el modelo queda **libre de escala**: no puede distinguir un par de 300 t de
uno de 5 kg y está obligado a predecirles el mismo ratio, cuando los grandes tienen ratios
estables y los chicos explosivos.

Se arregla con **una** feature, no sacando las siete del espacio relativo: `log_escala`, el log
de `s(t)`. Lleva la magnitud del par y sigue siendo una transformación de la escala, no una
tonelada cruda.

Y es la que más evidencia independiente tiene: **#1 por ganancia en las dos corridas del
selector, con el 17 % del total**. Se suman por el mismo criterio `cliente_rank_rel` (#2-3, 9 %
de la ganancia; coherente con que 13 clientes sean el 50,8 % del tonelaje) y `tn_prod_rel`
(#6-7). Total 18, por debajo del techo de 20.

Detalle: con los lags relativos, `nivel_relativo = lag_1/s` sería un duplicado exacto de
`lag1_rel`; se usa `nivel_rel = tn(t)/s`, que es el mes corriente. Y `ventas_ult12` va
**shifteada**, como en el z702. Los rangos de Optuna también vuelven a los suyos, más anchos que
los que yo había achicado: `lr` [0,005 – 0,2], `num_leaves` [16 – 256], `min_data_in_leaf`
[20 – 300], árboles [100 – 1500], 30 trials.

## Qué se conserva del z702 (las decisiones de modelado)

- **Clase = `tn(t+2)/s(t)`**, con `s` = media expandida causal y la convención `0/0 := 0`.
  El predict vuelve a toneladas con `pred = s · ŷ` antes de agregarse por producto.
- **Clusters DTW sobre las series de los pares**, que es la unidad que se modela.
- **15 features**, la lista cerrada del z702. Cada vez que agregué, el LB empeoró.
- **Validación anclada en febreros** para tunear. Los otros meses se miden pero **no eligen**.
- **Tweedie**, `variance_power` tuneado en [1,1 – 1,6], `max_bin = 1023`.
- **Sin multiplicador**: el submit va crudo. `m*` se sigue calculando y mostrando como
  diagnóstico, porque informa si el modelo tiene sesgo de nivel, pero no se aplica.

## Qué se corrige (sólo fugas y métrica, nada de modelado)

1. **El `.over(par)` va al final de la cadena.** En el z702 estaba adentro del `shift`, así que
   la ventana móvil corría sobre la columna entera y las primeras filas de cada serie se
   contaminaban con la cola de la serie anterior.
2. **El corte del fold.** Train con `periodo <= corte − h` y eval en `periodo == corte`. Con
   `periodo <= corte`, las filas de los últimos h meses traen como clase justamente el mes que
   se valida: es entrenar contra la respuesta.
3. **El WAPE se mide a nivel producto y global.** El z702 lo medía por cluster sobre sumas
   parciales del producto, que no es la métrica de Kaggle. De ahí sale un solo estudio de Optuna
   en vez de uno por cluster.
4. **Test de causalidad** que recalcula todo el FE sobre la historia truncada y **aborta** si
   alguna feature cambia. Ya cazó tres fugas reales mientras se escribía este pipeline.
5. **Clusters recalculados en cada corte**, con historia truncada. Calcularlos una vez sobre
   toda la serie hace que la etiqueta de cluster de una fila de 2018 dependa de 2019.

## Mejoras que sí entran sin salir de la lógica simple

**1. Cinco semillas en el ensamble final** (el z702 usaba dos). Promediar baja la varianza sin
tocar el sesgo, no agrega ni una feature y sólo encarece el entrenamiento final, no Optuna. Es
lo más parecido a una mejora gratis que queda.

**2. Reglas de post-proceso auto-evaluadas** (celda 10). Cuatro reglas de una línea —apagar
pares muertos, recortar predicciones absurdas respecto de la escala del par— evaluadas sobre las
predicciones de los folds **ya calculadas**, sin reentrenar. Se aplica **una sola**, y sólo si
gana por más de `margen_regla` **y** en la mayoría de los folds. En el smoke test ninguna llegó
—la mejor movió 0,0002— y el notebook predijo en crudo, que es el comportamiento correcto.
De paso eso descarta la hipótesis del "tonelaje fantasma": apagar los pares dormidos no cambia
nada, así que el modelo no los está inflando.

**3. Promediar `dividir` y `offset`** (dos corridas, cero código nuevo). Dieron 0,245 y 0,258 con
formulaciones genuinamente distintas, así que sus errores no están correlacionados del todo y el
promedio suele caer por debajo de ambos. Es el mismo argumento del multi-semilla pero con más
diversidad. Se hace corriendo el notebook dos veces con `modo_escala` distinto y promediando los
dos CSV.

**4. Probar sin submuestreo de dormidos** (`submuestreo_dormidos = 1.0`). Es insesgado en
esperanza pero agrega varianza, y con el peso de compensación multiplica por 4 el peso de filas
que son todas cero. Un flag, una corrida.

## Cómo seguir desde acá

Una sola cosa por vez, y que juzgue el leaderboard. El `offset`, el peso `s^(p−1)`, el objetivo
agrupado, las features estacionales y el peso por recencia siguen implementados y apagados
detrás de sus flags: si querés probar alguno, prendé **uno** y compará contra este.

Lo que ya no vale la pena volver a probar: el multiplicador.

In [ ]:
%pip install -q polars duckdb lightgbm optuna dtaidistance scikit-learn scipy pandas numpy pyarrow

In [ ]:
# ruff: noqa: E402
import gc
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAY_OPTUNA = True
except Exception:
    HAY_OPTUNA = False

try:
    from dtaidistance import dtw
    DTW_C = dtw.try_import_c()
except Exception:
    dtw, DTW_C = None, False

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/.drive")
    os.makedirs("/content/buckets", exist_ok=True)
    if not os.path.islink("/content/buckets/b1"):
        os.symlink("/content/.drive/My Drive/labo3", "/content/buckets/b1")
    os.environ["LABO3_BUCKET"] = "/content/buckets/b1"

N_CORES = os.cpu_count() or 8


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env); p.mkdir(parents=True, exist_ok=True); return p
    for c in ("/home/jupyter/buckets/b1", "/content/buckets/b1", "~/buckets/b1"):
        p = Path(c).expanduser()
        if p.exists():
            return p
    p = Path.cwd() / "bucket"; p.mkdir(parents=True, exist_ok=True); return p


BUCKET = resolver_bucket()


def escribir_atomico(fn, destino: Path):
    """Escribe por un temporal y copia. El rename sobre un bucket montado por FUSE no es confiable."""
    tmp = Path("/tmp") / (destino.name + ".tmp")
    fn(tmp)
    import shutil
    shutil.copyfile(tmp, destino)
    tmp.unlink(missing_ok=True)


def meses_entre(a: int, b: int) -> int:
    return (b // 100 - a // 100) * 12 + (b % 100 - a % 100)


def periodo_menos(p: int, k: int) -> int:
    a, m = divmod(p, 100)
    t = a * 12 + (m - 1) - k
    return (t // 12) * 100 + (t % 12) + 1


print(f"bucket: {BUCKET} | cores: {N_CORES} | optuna: {HAY_OPTUNA} | DTW en C: {DTW_C}")

## 1 — Palancas

In [ ]:
PARAM = {
    "experimento": "z711_vuelta_z702",
    "kaggle_competition": "labo-iii-2026-ba",
    "modo_test": False,          # True = smoke test con pocos productos

    "horizonte": 2,
    "periodo_inferencia": 201912,
    "periodo_objetivo": 202002,

    # --- escalado (relativo al mes donde estás parado, causal) ---
    "escala": "media_expandida",   # media_expandida | media_movil_12 | mediana_movil_12
    "piso_escala": 1e-3,

    # --- objetivo ---
    # 'tweedie'  : init_score=log(s) + peso s^(p-1)  -> la hoja vale Sum(y)/Sum(s)   [recomendado]
    # 'agrupado' : objetivo custom cuyo gradiente es el signo del error AGREGADO por producto,
    #              que es literalmente el numerador de WAPE. Experimental: A/B en la celda 12.
    # 'dividir': clase = tn(t+2)/s(t), pred = s*yhat   <- lo del z702, que dio 0.245 en el LB
    # 'offset' : init_score = log(s), pred = s*exp(f)    <- z705 (0.258) y z709 (0.314). Peor.
    "modo_escala": "dividir",
    "objetivo": "tweedie",
    "tweedie_power": 1.3,          # arranque; Optuna lo tunea y el peso se reajusta con set_weight
    "usar_peso_escala": False,     # el s^(p-1) es del modo offset; el z702 no pesaba

    # --- features (24). Criterio, no canarito: se sacaron las que quedaron al fondo en las
    #     dos corridas del selector (meses_desde_compra, racha_max_12, tasa_actividad_12,
    #     croston_intervalo, sku_size) y se sumaron las tres estacionales nuevas. ---
    # Las 15 del z702, ni una mas. Cada vez que agregue features el LB empeoro:
    # 15 -> 0.245 (z702), 20 -> 0.258 (z705), 24 -> 0.314 (z709).
    # 18 features, TODAS escaladas o adimensionales — la consigna de la catedra se respeta.
    #
    # El problema que resolvia poner los lags en crudo era otro: con el target dividido y las
    # features tambien divididas el modelo queda LIBRE DE ESCALA y no puede distinguir un par de
    # 300 t de uno de 5 kg, asi que esta obligado a predecirles el mismo ratio. Eso se arregla con
    # UNA feature en vez de siete: `log_escala`, que es el log de s(t) — escalado, no crudo — y le
    # da la magnitud del par sin sacar nada del espacio relativo.
    #
    # `log_escala` ademas es la que mas evidencia independiente tiene: #1 por ganancia en las dos
    # corridas del selector, con el 17% del total. `cliente_rank_rel` salio #2-3 (9%), coherente
    # con que los 13 clientes mas grandes sean el 50,8% del tonelaje. `tn_prod_rel` salio #6-7.
    #
    # Nota: con los lags relativos, `nivel_relativo = lag_1/s` seria un duplicado exacto de
    # `lag1_rel`. Se usa `nivel_rel = tn(t)/s`, que es el mes corriente y si aporta.
    "features": [
        # dinamica del par, relativa a su propia escala
        "lag1_rel", "lag2_rel", "lag3_rel", "lag12_rel",
        "rmean3_rel", "rmean12_rel", "rstd3_rel",
        "nivel_rel", "tendencia_3_12",
        # intermitencia (adimensionales)
        "meses_consec_sin_compra", "ventas_ult12",
        # magnitud del par: la unica que lleva tamano, y en log
        "log_escala",
        # calendario y contexto
        "mes", "share_prod_en_cat3", "tn_prod_rel", "cliente_rank_rel",
        # estaticas
        "cat3", "brand",
    ],
    "categoricas": ["cat3", "brand"],
    "usar_mes_crudo": True,    # el z702 tenia `mes`. `mes_objetivo` no existe en su lista

    # --- clusters DTW sobre las series de los PARES (la unidad que se modela) ---
    "usar_clusters": True,
    "cl_lista_k": [4, 5, 6, 7, 8],
    "cl_ventana": 24,
    "cl_banda": 3,                 # Sakoe-Chiba, honesta porque la ventana es comun
    "cl_balance_min": 0.03,
    "cl_min_meses_activos": 6,     # menos que esto -> cluster "intermitente", no se le hace DTW
    "cl_muestra": 3000,            # series para la matriz DTW completa; el resto va por medoide

    # --- validacion walk-forward repartida en el ano ---
    # Optuna usa 3 anclas que cubren los tres regimenes: diciembre (origen OCTUBRE, el caso
    # dificil), febrero (el target) y agosto (mes flojo). El reporte final mide en las 7.
    # Optuna NO ve ningun febrero: asi el multiplicador se estima en anclas limpias, incluidas
    # las dos de febrero, sin la optimista de haber tuneado contra ellas.
    # El z702 validaba "por febreros" y dio 0.245; el z709 repartio anclas por todo el ano y dio
    # 0.314. Se vuelve a febrero para TUNEAR, y los otros meses quedan solo como reporte: sirven
    # para saber si generaliza, no para elegir.
    "anclas_optuna": [201802, 201902],
    "anclas_reporte": [201802, 201810, 201902, 201906, 201910],
    "peso_febrero": 1.0,   # ya todas las anclas de tuneo son de febrero

    # --- entrenamiento ---
    "max_bin": 1023,
    "techo_arboles": 1500,   # el z702 buscaba n_estimators en [100, 1500]
    "n_trials": 30,
    "submuestreo_dormidos": 0.25,
    # 5 semillas. Es la unica mejora gratis que queda: promediar reduce la varianza sin tocar el
    # sesgo, no agrega ni una feature, y solo encarece el entrenamiento FINAL (no Optuna).
    "semillas_ensemble": [102191, 314159, 777773, 116269, 241511],
    "semillas_reporte": [102191, 314159],   # el reporte usa ensemble, como el modelo que se sube
    # Vida media en meses del peso por recencia (Optuna la tunea). Valores altos = casi sin
    # decaimiento; el rango deja que decida si conviene mirar mas lo reciente o todo por igual.
    "vida_media_rango": None,   # None = sin peso por recencia (el z702 no tenia)

    # --- calibracion ---
    # NO se aplica multiplicador. Siete puntos de LB, dos pipelines distintos, cero excepciones:
    #   z702  x0.9 0.252 | x1.0 0.245 | x1.1 0.268
    #   z705  m0.848 0.297 | m0.970 0.262 | m1.000 0.258 | m1.092 0.260
    # Se sigue estimando y mostrando m* como diagnostico, pero el submit va en crudo.
    "aplicar_multiplicador": False,
    "margen_regla": 0.002,   # una regla de post-proceso tiene que ganar esto para aplicarse
    "lambda_shrink": 0.7,
    "banda_nivel": [28400, 30600],  # feb-2020 plausible; fuera de esto se avisa fuerte
    "submit": True,
    "sufijo": "",

    # Versiona los checkpoints. Si cambia la densa, el FE o la lista de features, SUBILA: los
    # checkpoints viejos quedan ignorados en vez de reusarse en silencio con otra logica.
    # v1 = pares observados (10,5M). v2 = cartesiano catedra (17,2M). v3 = 18 features escaladas.
    "version_datos": "v3_feats18",   # cambio la lista de features -> checkpoints nuevos
}

if PARAM["objetivo"] == "agrupado":
    # sin filas no se puede agregar bien, y el peso s^(p-1) alinea una perdida por fila que
    # este objetivo ya no usa
    PARAM.update(submuestreo_dormidos=1.0, usar_peso_escala=False)
    print("objetivo AGRUPADO: se apagan submuestreo de dormidos y peso s^(p-1)")

if PARAM["modo_test"]:
    PARAM.update(n_trials=3, techo_arboles=120, semillas_ensemble=[102191],
                 anclas_optuna=[201905], anclas_reporte=[201812, 201902],
                 semillas_reporte=[102191],
                 cl_lista_k=[3, 4], submit=False)

DIR_RAW = BUCKET / "datasets"
DIR_EXP = BUCKET / "exp" / (PARAM["experimento"] + PARAM["sufijo"])
DIR_CK = DIR_EXP / "ck" / PARAM["version_datos"]
DIR_OUT = DIR_EXP / "submits"
for d in (DIR_RAW, DIR_EXP, DIR_CK, DIR_OUT):
    d.mkdir(parents=True, exist_ok=True)

FEATS = list(PARAM["features"])
if not PARAM["usar_mes_crudo"]:
    FEATS = [f for f in FEATS if f not in ("mes", "mes_objetivo")]
CATEGORICAS = [c for c in PARAM["categoricas"] if c in FEATS]
print(f"experimento: {DIR_EXP}")
print(f"checkpoints: {DIR_CK}  (version {PARAM['version_datos']})")
print(f"features ({len(FEATS)}): {FEATS}")

## 2 — Densa zero-fill (criterio cátedra, **producto cartesiano**)

`todos los clientes × todos los productos × todos los periodos`, recortado a la vida del producto
(`min..max` observado) y a partir del primer periodo del cliente. **17.173.448 filas.**

Es el `tb_zeroes` del `z601` de la cátedra, escrito como `CROSS JOIN` + `LEFT JOIN` en vez de
`NOT EXISTS` + `UNION` porque es mucho más rápido y da lo mismo.

**Por qué el cartesiano y no sólo los pares que comerciaron** (que serían 10.469.949, un 61 %):
no es sólo fidelidad al criterio. Quedarse con los pares observados obliga a definir el universo
de filas con `DISTINCT (cliente, producto)` sobre **toda** la historia, futuro incluido: en un
fold cortado en 201810 entrarían pares cuyo primer intercambio ocurre recién en 201905. Las
features no se contaminan —esas filas son ceros— pero la *composición del train* pasa a depender
del futuro, y eso un deploy real no lo puede saber. El cartesiano es libre de fuga por
construcción. Además permite predecir la activación de pares nuevos, que es el **1,6 % del
tonelaje mensual** (1,9 % en los últimos seis meses).

Los 6,7 M de filas de más son casi todas de pares dormidos, así que el submuestreo al 25 % y el
peso `s^(p−1)` las dejan pesando poco: el costo real es de orden +20 %, no +64 %.

Los meses quedan **contiguos**, y hay un assert que lo verifica en vez de darlo por sentado: sin
contigüidad, `shift(k)` deja de ser el lag de k meses y todos los lags mienten.

In [ ]:
def descargar(a: str):
    dst = DIR_RAW / a
    if dst.exists():
        return
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{a}"
    subprocess.run(["wget", "-q", url, "-O", str(dst)], check=True)


for _a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"):
    descargar(_a)

PROD_TARGET = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt",
                          separator="\t")["product_id"].cast(pl.Int64).to_list()

CK_DENSA = DIR_CK / "densa.parquet"
if CK_DENSA.exists():
    densa = pl.read_parquet(CK_DENSA)
    print(f"densa desde checkpoint {CK_DENSA.parent.name}: {densa.height:,} filas")
else:
    t0 = time.time()
    con = duckdb.connect()
    con.execute("PRAGMA threads=%d" % N_CORES)
    con.execute(f"""CREATE OR REPLACE VIEW crudo AS
        SELECT customer_id, product_id, periodo, sum(tn) AS tn
        FROM read_csv('{DIR_RAW / "sell-in.txt.gz"}', delim='\t', header=true)
        GROUP BY 1,2,3""")
    filtro = ""
    if PARAM["modo_test"]:
        filtro = f"WHERE product_id IN ({','.join(map(str, PROD_TARGET[:60]))})"
    densa = con.execute(f"""
    WITH cal AS (SELECT DISTINCT periodo FROM crudo),
         vida AS (SELECT product_id, min(periodo) p0, max(periodo) p1 FROM crudo
                  {filtro} GROUP BY 1),
         ini  AS (SELECT customer_id, min(periodo) c0 FROM crudo GROUP BY 1)
    SELECT i.customer_id, v.product_id, c.periodo, coalesce(s.tn, 0.0) AS tn
    FROM vida v
    CROSS JOIN ini i
    JOIN cal c ON c.periodo BETWEEN v.p0 AND v.p1 AND c.periodo >= i.c0
    LEFT JOIN crudo s ON s.customer_id = i.customer_id
                     AND s.product_id = v.product_id
                     AND s.periodo = c.periodo
    """).pl()
    prods = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
             .select(["product_id", "cat2", "cat3", "brand", "sku_size"])
             .unique(subset=["product_id"]))
    densa = (densa.join(prods, on="product_id", how="left")
             .with_columns((pl.col("customer_id").cast(pl.Utf8) + "_"
                            + pl.col("product_id").cast(pl.Utf8)).alias("par"))
             .sort(["par", "periodo"]))
    escribir_atomico(lambda p: densa.write_parquet(p), CK_DENSA)
    print(f"densa: {densa.height:,} filas en {time.time() - t0:.0f}s -> {CK_DENSA}")

_m = densa.group_by("par").agg([pl.len().alias("n"), pl.col("periodo").min().alias("a"),
                                pl.col("periodo").max().alias("b")])
_m = _m.with_columns(((pl.col("b") // 100 - pl.col("a") // 100) * 12
                      + (pl.col("b") % 100 - pl.col("a") % 100) + 1).alias("esp"))
assert (_m["n"] == _m["esp"]).all(), "hay huecos de periodo: shift(k) dejaria de ser el lag k"
print(f"grid contiguo verificado | pares: {densa['par'].n_unique():,} | "
      f"productos: {densa['product_id'].n_unique():,}")
if not PARAM["modo_test"]:
    _esp = 17_173_448   # criterio catedra z601, cartesiano completo
    if densa.height != _esp:
        print(f"\n>>> ATENCION: la densa tiene {densa.height:,} filas y el criterio catedra da "
              f"{_esp:,}.\n>>> Si dice 10,469,949 estas levantando un checkpoint viejo (solo pares "
              f"observados).\n>>> Subi PARAM['version_datos'] o borra {DIR_CK}.\n")
del _m; gc.collect()

## 3 — Escalado y feature engineering

`s(t)` es **relativo al mes donde estás parado**: media expandida de `tn` hasta t **inclusive**.
Nunca toca `t+1` ni `t+2`. Las alternativas (`media_movil_12`, `mediana_movil_12`) son también
ventanas que terminan en t.

Regla de oro de polars: el `.over(par)` va **al final** de la cadena. Si va adentro del `shift`,
la ventana móvil corre sobre la columna entera y se derrama de la cola de la serie anterior — eso
es fuga, y era un bug real del z702.

In [ ]:
def _escala(d: pl.DataFrame, g: str = "par") -> pl.DataFrame:
    tn = pl.col("tn")
    if PARAM["escala"] == "media_movil_12":
        e = tn.rolling_mean(12, min_samples=1).over(g)
    elif PARAM["escala"] == "mediana_movil_12":
        e = tn.rolling_median(12, min_samples=1).over(g)
    else:
        e = tn.cum_sum().over(g) / pl.int_range(1, pl.len() + 1).over(g)
    return d.with_columns(e.alias("escala"))


def _fe(d: pl.DataFrame) -> pl.DataFrame:
    g = "par"
    tn = pl.col("tn")
    d = _escala(d, g)
    esc = pl.max_horizontal(pl.col("escala"), pl.lit(PARAM["piso_escala"]))

    eps = 1e-6
    d = d.with_columns([
        # crudas, en toneladas: es lo que le da al modelo la nocion de tamano del par
        tn.shift(1).over(g).alias("lag_1"),
        tn.shift(2).over(g).alias("lag_2"),
        tn.shift(3).over(g).alias("lag_3"),
        tn.shift(12).over(g).alias("lag_12"),
        tn.shift(1).rolling_mean(3, min_samples=1).over(g).alias("rmean_3"),
        tn.shift(1).rolling_mean(12, min_samples=1).over(g).alias("rmean_12"),
        tn.shift(1).rolling_std(3, min_samples=2).over(g).alias("rstd_3"),
        # shifteada: no mira el mes actual, igual que el z702
        (tn > 0).cast(pl.Int32).rolling_sum(12, min_samples=1).shift(1).over(g)
            .alias("ventas_ult12"),
        (pl.col("periodo") % 100).alias("mes"),
        pl.col("escala").log1p().alias("log_escala"),
        (tn > 0).cast(pl.Int32).alias("_hay"),
        # variantes relativas, por si se quieren probar (no estan en la lista de 15)
        (tn / esc).alias("nivel_rel"),
        (tn.shift(1).over(g) / esc).alias("lag1_rel"),
        (tn.shift(2).over(g) / esc).alias("lag2_rel"),
        (tn.shift(3).over(g) / esc).alias("lag3_rel"),
        (tn.shift(12).over(g) / esc).alias("lag12_rel"),
        (tn.shift(1).rolling_std(3, min_samples=2).over(g) / esc).alias("rstd3_rel"),
        (tn.shift(1).rolling_mean(3, min_samples=1).over(g) / esc).alias("rmean3_rel"),
        (tn.shift(1).rolling_mean(12, min_samples=1).over(g) / esc).alias("rmean12_rel"),
    ])
    d = d.with_columns([
        (pl.col("lag_1") / (pl.col("escala") + eps)).alias("nivel_relativo"),
        (pl.col("rmean_3") / (pl.col("rmean_12") + eps)).alias("tendencia_3_12"),
    ])

    # meses consecutivos sin compra, con el forward-fill del ultimo indice con venta (z702)
    d = d.with_columns(pl.int_range(pl.len()).over(g).alias("_rn"))
    d = d.with_columns(
        pl.when(tn > 0).then(pl.col("_rn")).otherwise(None).forward_fill().over(g).alias("_uv"))
    d = d.with_columns(
        (pl.col("_rn") - pl.col("_uv").fill_null(-1)).alias("meses_consec_sin_compra"))
    d = d.with_columns(pl.col("meses_consec_sin_compra").alias("meses_desde_compra"))

    # contexto: producto, cliente, categoria y familia en el mismo periodo (todo <= t)
    d = d.with_columns([
        pl.col("tn").sum().over(["product_id", "periodo"]).alias("_tn_prod"),
        pl.col("tn").sum().over(["customer_id", "periodo"]).alias("_tn_cli"),
        pl.col("tn").sum().over(["cat3", "periodo"]).alias("_tn_cat3"),
        pl.col("tn").sum().over(["cat3", "brand", "customer_id", "periodo"]).alias("_tn_fam"),
    ])
    d = d.with_columns([
        (pl.col("_tn_prod") / (pl.col("_tn_cat3") + eps)).alias("share_prod_en_cat3"),
        (pl.col("_tn_prod").shift(1).over(g) / esc).alias("tn_prod_rel"),
        (tn / (pl.col("_tn_prod") + eps)).alias("share_par_en_prod"),
        (tn / (pl.col("_tn_cli") + eps)).alias("share_par_en_cli"),
        (pl.col("_tn_fam").rolling_sum(3, min_samples=1).over(g) / (3 * esc))
            .alias("stock_idx_familia_3"),
        (pl.col("_tn_prod").shift(1).rolling_mean(3, min_samples=1).over(g)
         / (pl.col("_tn_prod").shift(1).rolling_mean(12, min_samples=1).over(g) + eps)
         ).alias("tendencia_prod_3_12"),
    ])

    # rank del cliente por volumen acumulado hasta t, sobre una tabla ordenada POR PERIODO
    cli = (d.group_by(["customer_id", "periodo"]).agg(pl.col("tn").sum().alias("_v"))
           .sort(["customer_id", "periodo"])
           .with_columns(pl.col("_v").cum_sum().over("customer_id").alias("_acum")))
    cli = cli.with_columns((pl.col("_acum").rank("average").over("periodo")
                            / pl.len().over("periodo")).alias("cliente_rank_rel"))
    d = d.join(cli.select(["customer_id", "periodo", "cliente_rank_rel"]),
               on=["customer_id", "periodo"], how="left")

    d = _estacionalidad(d)
    return d.drop([c for c in d.columns if c.startswith("_")]).sort(["par", "periodo"])


def _estacionalidad(d: pl.DataFrame) -> pl.DataFrame:
    """Estacionalidad del MES OBJETIVO y avance anual del producto. Todo con datos `<= t`.

    Por que importa: el sell-in tiene indice 1.15 en octubre y 0.84 en diciembre. Con horizonte 2,
    el modelo parado en octubre ve su propio nivel inflado y tiene que predecir el piso del ano.
    Sin decirle cuanto pesa historicamente el mes que va a predecir, sobre-predice ~37%.

    La causalidad la garantiza el `join_asof` hacia atras: para una fila de `t` que apunta al mes
    `m*`, toma la ultima ocurrencia de `m*` con `periodo <= t`. Si `m*` todavia no ocurrio nunca,
    queda 1.0 (neutro).
    """
    h = PARAM["horizonte"]
    d = d.with_columns(((((pl.col("periodo") % 100) - 1 + h) % 12) + 1).cast(pl.Int32)
                       .alias("mes_objetivo"))

    pp = (d.group_by(["product_id", "periodo"]).agg(pl.col("tn").sum().alias("tnp"))
          .with_columns([(pl.col("periodo") % 100).cast(pl.Int32).alias("mes"),
                         (pl.col("periodo") // 100).cast(pl.Int32).alias("anio")]))

    # media expandida del producto para ESE mes calendario, y media expandida global
    pm = (pp.sort(["product_id", "mes", "periodo"])
          .with_columns((pl.col("tnp").cum_sum().over(["product_id", "mes"])
                         / pl.int_range(1, pl.len() + 1).over(["product_id", "mes"]))
                        .alias("_m_mes"))
          .select(["product_id", "mes", "periodo", "_m_mes"]).sort("periodo"))
    pg = (pp.sort(["product_id", "periodo"])
          .with_columns((pl.col("tnp").cum_sum().over("product_id")
                         / pl.int_range(1, pl.len() + 1).over("product_id")).alias("_m_glob"))
          .select(["product_id", "periodo", "_m_glob"]))

    izq = (d.select(["product_id", "periodo", "mes_objetivo"]).unique()
           .with_columns(pl.col("mes_objetivo").alias("mes")).sort("periodo"))
    est = (izq.join_asof(pm, on="periodo", by=["product_id", "mes"], strategy="backward")
           .join(pg, on=["product_id", "periodo"], how="left")
           .with_columns((pl.col("_m_mes") / (pl.col("_m_glob") + 1e-9))
                         .fill_null(1.0).fill_nan(1.0).alias("estacional_prod_objetivo"))
           .select(["product_id", "periodo", "estacional_prod_objetivo"]))

    # avance anual del producto: acumulado del ano hasta t contra el mismo tramo del ano anterior
    ytd = (pp.sort(["product_id", "anio", "mes"])
           .with_columns(pl.col("tnp").cum_sum().over(["product_id", "anio"]).alias("_ytd")))
    prev = ytd.select([pl.col("product_id"), (pl.col("anio") + 1).alias("anio"),
                       pl.col("mes"), pl.col("_ytd").alias("_ytd_prev")])
    ytd = (ytd.join(prev, on=["product_id", "anio", "mes"], how="left")
           .with_columns((pl.col("_ytd") / (pl.col("_ytd_prev") + 1e-9))
                         .fill_null(1.0).fill_nan(1.0).alias("avance_anual_prod"))
           .select(["product_id", "periodo", "avance_anual_prod"]))

    return (d.join(est, on=["product_id", "periodo"], how="left")
            .join(ytd, on=["product_id", "periodo"], how="left")
            .with_columns([pl.col("estacional_prod_objetivo").fill_null(1.0),
                           pl.col("avance_anual_prod").fill_null(1.0)]))


CK_FE = DIR_CK / "fe.parquet"
if CK_FE.exists():
    datos = pl.read_parquet(CK_FE)
    faltan = [c for c in FEATS if c not in datos.columns]
    if faltan:
        raise RuntimeError(
            f"El checkpoint de FE no tiene {faltan}. Subi PARAM['version_datos'] "
            f"o borra {CK_FE}.")
    print(f"FE desde checkpoint {CK_FE.parent.name}: {datos.height:,} filas")
else:
    t0 = time.time()
    datos = _fe(densa).with_columns(
        pl.col("tn").shift(-PARAM["horizonte"]).over("par").alias("clase"))
    cols = ["par", "customer_id", "product_id", "periodo", "tn", "escala", "clase"] + FEATS
    datos = datos.select(list(dict.fromkeys(cols)))
    escribir_atomico(lambda p: datos.write_parquet(p), CK_FE)
    print(f"FE: {datos.height:,} filas x {len(datos.columns)} cols en {time.time() - t0:.0f}s")

## 4 — Test de causalidad (el notebook aborta si falla)

Recalcula el FE **entero** sobre la historia truncada en T y compara contra el FE calculado sobre
toda la historia y después filtrado a T. Si una sola feature difiere, esa feature usó información
de después de T. No hay excusa posible: aborta.

In [ ]:
def test_causalidad(T: int = 201806, n_pares: int = 800):
    pares = densa["par"].unique().sort().head(n_pares).to_list()
    sub = densa.filter(pl.col("par").is_in(pares))
    a = _fe(sub).filter(pl.col("periodo") <= T).sort(["par", "periodo"])
    b = _fe(sub.filter(pl.col("periodo") <= T)).sort(["par", "periodo"])
    assert a.height == b.height, f"distinta cantidad de filas: {a.height} vs {b.height}"
    malas = []
    for c in FEATS + ["escala"]:
        x, y = a[c], b[c]
        if x.dtype == pl.Utf8 or c in CATEGORICAS:
            if not (x.to_numpy() == y.to_numpy()).all():
                malas.append(c)
            continue
        xn, yn = x.to_numpy().astype(float), y.to_numpy().astype(float)
        if not np.allclose(np.nan_to_num(xn), np.nan_to_num(yn), rtol=1e-9, atol=1e-9):
            malas.append(c)
    if malas:
        raise AssertionError(
            f"FUGA DE FUTURO en: {malas}. El FE de esas columnas usa informacion de despues de T.")
    print(f"test de causalidad OK: {len(FEATS) + 1} columnas, T={T}, {n_pares} pares")


test_causalidad()
test_causalidad(T=201903, n_pares=800)

## 5 — Clusters DTW a nivel producto

Se clusteriza la serie mensual de cada **producto** (agregando clientes), que es el nivel al que
mide Kaggle. Los pares heredan el cluster de su producto: el dataset queda particionado, y como
ningún producto se reparte entre clusters, `Σ_p |e_p|` se descompone exacto por cluster.

Ventana común de 24 meses con padding a izquierda para que la banda Sakoe-Chiba de 3 sea honesta
(el z702 usaba `max(3, |Δlargo|)`, que en pares de largos dispares se abría hasta desactivarse).
`k` se elige por silhouette con restricción de balance sobre `[4..8]` — la cátedra espera 6/7,
pero se elige, no se hardcodea.

**Se recalcula en cada corte temporal.** Si se calculara una vez sobre toda la serie, la etiqueta
de cluster de una fila de 2018 estaría condicionada por lo que pasó en 2019: fuga sutil y real.

In [ ]:
def _series_pares(corte: int):
    """Matriz de series normalizadas de los pares CON SENAL, y la lista de pares intermitentes.

    Se normaliza cada serie (z-score) para que DTW compare FORMA y no volumen: el volumen ya
    entra al modelo por `log_escala`, y sin normalizar el clustering degenera en "grandes contra
    chicos". Ventana comun de `cl_ventana` meses terminando en `corte`, con padding a izquierda,
    para que la banda de Sakoe-Chiba sea honesta.
    """
    W = PARAM["cl_ventana"]
    desde = periodo_menos(corte, W - 1)
    d = (datos.filter((pl.col("periodo") <= corte) & (pl.col("periodo") >= desde))
         .select(["par", "periodo", "tn"]).sort(["par", "periodo"]))
    ag = d.group_by("par").agg([(pl.col("tn") > 0).sum().alias("act"),
                                pl.col("tn").sum().alias("vol")])
    vivos = ag.filter(pl.col("act") >= PARAM["cl_min_meses_activos"])
    inter = ag.filter(pl.col("act") < PARAM["cl_min_meses_activos"])["par"].to_list()

    piv = (d.filter(pl.col("par").is_in(vivos["par"].to_list()))
           .pivot(values="tn", index="par", on="periodo", aggregate_function="sum")
           .fill_null(0.0).sort("par"))
    cols = [c for c in piv.columns if c != "par"]
    M = piv.select(cols).to_numpy().astype(np.double)
    if M.shape[1] < W:                                  # padding a izquierda
        M = np.hstack([np.zeros((len(M), W - M.shape[1])), M])
    mu, sd = M.mean(1, keepdims=True), M.std(1, keepdims=True)
    M = (M - mu) / np.where(sd > 1e-9, sd, 1.0)
    pares = piv["par"].to_list()
    vol = vivos.sort("par")["vol"].to_numpy()
    return pares, M, vol, inter


def _matriz_dtw(M: np.ndarray) -> np.ndarray:
    n = len(M)
    if dtw is not None:
        D = np.asarray(dtw.distance_matrix_fast(M, window=PARAM["cl_banda"],
                                                parallel=True, compact=False))
        iu = np.triu_indices(n, 1)
        v = D[iu]
        fin = np.isfinite(v)
        if not fin.all():                               # banda infactible -> sanear, no propagar inf
            v = np.where(fin, v, (v[fin].max() if fin.any() else 1.0) * 10)
        F = np.zeros((n, n)); F[iu] = v
        return F + F.T
    print("   dtaidistance no disponible -> euclidea sobre la forma (fallback)")
    return np.sqrt(((M[:, None, :] - M[None, :, :]) ** 2).sum(-1))


def clusters_en(corte: int) -> pl.DataFrame:
    """Devuelve (par, cl). Causal: sólo mira `<= corte`, y se recalcula en cada fold.

    Con ~736k pares la matriz DTW completa es inviable (2,7e11 pares), así que:
      1. se toma una MUESTRA estratificada por volumen de los pares con senal,
      2. matriz DTW completa sobre la muestra -> jerarquico -> k por silhouette + balance,
      3. se extrae el MEDOIDE de cada cluster,
      4. cada par restante se asigna al medoide mas cercano por DTW.
    Los pares intermitentes (menos de `cl_min_meses_activos` meses con venta) NO se clusterizan:
    van a un cluster propio. Son el 61% de los pares y el 6% del tonelaje; hacerles DTW es medir
    ruido y es lo que hace que todo colapse en un cluster dominante.
    """
    ck = DIR_CK / f"clusters_{corte}.parquet"
    if ck.exists():
        return pl.read_parquet(ck)
    t0 = time.time()
    pares, M, vol, inter = _series_pares(corte)
    if not PARAM["usar_clusters"] or len(pares) < 50:
        out = pl.DataFrame({"par": pares + inter, "cl": [0] * (len(pares) + len(inter))})
        escribir_atomico(lambda f: out.write_parquet(f), ck)
        return out

    # 1) muestra estratificada por volumen (los grandes son los que mueven WAPE)
    n_m = min(PARAM["cl_muestra"], len(pares))
    orden = np.argsort(-vol)
    top = orden[: n_m // 2]
    resto = np.setdiff1d(orden[n_m // 2:], top, assume_unique=False)
    rng = np.random.default_rng(102191)
    sel = np.concatenate([top, rng.choice(resto, min(n_m - len(top), len(resto)), replace=False)])
    Ms = np.ascontiguousarray(M[sel])

    # 2) matriz completa sobre la muestra -> jerarquico -> k
    D = _matriz_dtw(Ms)
    Z = linkage(squareform(D, checks=False), method="average")
    cands = []
    for k in PARAM["cl_lista_k"]:
        lab = fcluster(Z, k, criterion="maxclust")
        if len(set(lab)) < 2:
            continue
        bal = np.bincount(lab)[1:].min() / len(lab)
        cands.append({"k": k, "lab": lab, "bal": bal,
                      "sil": silhouette_score(D, lab, metric="precomputed")})
    ok = [c for c in cands if c["bal"] >= PARAM["cl_balance_min"]]
    best = max(ok, key=lambda c: c["sil"]) if ok else max(cands, key=lambda c: c["bal"])
    K_forma = best["k"]

    # 3) medoide de cada cluster de la muestra
    med = []
    for k in range(1, K_forma + 1):
        idx = np.where(best["lab"] == k)[0]
        med.append(Ms[idx[np.argmin(D[np.ix_(idx, idx)].sum(1))]])
    med = np.ascontiguousarray(np.array(med))

    # 4) asignar TODOS los pares con senal al medoide mas cercano
    if dtw is not None:
        dist = np.empty((len(M), K_forma))
        for j in range(K_forma):
            dist[:, j] = [dtw.distance_fast(np.ascontiguousarray(x), med[j],
                                            window=PARAM["cl_banda"]) for x in M]
    else:
        dist = np.sqrt(((M[:, None, :] - med[None, :, :]) ** 2).sum(-1))
    asign = dist.argmin(1)

    out = pl.DataFrame({"par": pares + inter,
                        "cl": np.concatenate([asign, np.full(len(inter), K_forma)]).astype(np.int32)})
    escribir_atomico(lambda f: out.write_parquet(f), ck)
    tam = np.bincount(out["cl"].to_numpy(), minlength=K_forma + 1).tolist()
    print(f"   corte {corte}: k_forma={K_forma} (+1 intermitente) silhouette={best['sil']:.3f} "
          f"muestra={len(sel)} tamanos={tam} ({time.time() - t0:.0f}s)")
    return out


CORTES = sorted({periodo_menos(a, PARAM["horizonte"])
                 for a in PARAM["anclas_optuna"] + PARAM["anclas_reporte"]}
                | {PARAM["periodo_inferencia"]})
print("calculando clusters por corte (causal, DTW sobre las series de los pares):")
CLUSTERS = {c: clusters_en(c) for c in CORTES}
K = int(max(m["cl"].max() for m in CLUSTERS.values())) + 1
print(f"K = {K}  (el ultimo es el cluster intermitente)")

## 6 — Matrices, entrenamiento y métrica

**El peso.** `w = s^(p−1)`, y para las filas de pares dormidos submuestreadas al 25 %, un factor
`1/0,25` que las devuelve a su peso original — el estimador sigue siendo el de la misma pérdida,
con la memoria dividida.

**La métrica** es WAPE agregando a **producto**: se suman las predicciones de todos los pares del
producto *antes* de medir. Los errores entre clientes del mismo producto se cancelan, igual que
en Kaggle.

In [ ]:
COD = {c: {v: i for i, v in enumerate(sorted(datos[c].unique().drop_nulls().to_list()))}
       for c in CATEGORICAS}
print("categoricas:", {c: len(v) for c, v in COD.items()})
CAT_NOM = list(CATEGORICAS)


def paquete(corte: int, modo: str, semilla: int, mapa: pl.DataFrame) -> dict:
    """Parado en `corte` (ultimo mes con datos), con horizonte h.

    modo='train': filas con `periodo <= corte - h`. Es la unica ventana cuya clase `tn(t+h)`
        ya ocurrio en `<= corte`. Si se tomara `periodo <= corte`, las filas de los ultimos h
        meses traerian como clase justamente el mes que se quiere predecir: fuga directa.
    modo='eval' : filas de `periodo == corte`, cuya clase es `tn(corte + h)` — el mes objetivo.
    """
    h = PARAM["horizonte"]
    d = (datos.filter(pl.col("periodo") == corte) if modo == "eval"
         else datos.filter((pl.col("periodo") <= periodo_menos(corte, h))
                           & pl.col("clase").is_not_null()))
    d = d.join(mapa, on="par", how="left").with_columns(pl.col("cl").fill_null(0))
    cols = list(dict.fromkeys(["product_id", "periodo", "escala", "clase", "ventas_ult12", "cl"]
                              + FEATS))
    pdf = d.select(cols).to_pandas()
    for c in CATEGORICAS:
        if c in pdf.columns:
            pdf[c] = pdf[c].map(COD[c]).fillna(-1).astype("int32")

    s = np.maximum(pdf["escala"].to_numpy(), PARAM["piso_escala"])
    y = pdf["clase"].to_numpy()
    prod = pdf["product_id"].to_numpy()
    w = np.ones_like(s)   # s^(p-1) y la recencia se aplican al entrenar, con los params del trial
    edad = np.zeros_like(s) if modo == "eval" else np.array(
        [meses_entre(int(x), corte) for x in pdf["periodo"].to_numpy()], dtype=np.float32)

    if modo == "train":
        # el corte ya garantiza que ninguna clase de entrenamiento cae despues de `corte`
        assert pdf["periodo"].max() + 0 <= periodo_menos(corte, PARAM["horizonte"]), \
            "hay filas de train cuya clase cae despues del corte"
        if PARAM["submuestreo_dormidos"] < 1.0:
            rng = np.random.default_rng(semilla)
            dormida = (pdf["ventas_ult12"].to_numpy() == 0) & (y == 0)
            tasa = PARAM["submuestreo_dormidos"]
            keep = ~dormida | (rng.random(len(y)) < tasa)
            w = np.where(dormida, w / tasa, w)
            pdf, y, s, w, prod, edad = (pdf[keep], y[keep], s[keep], w[keep],
                                        prod[keep], edad[keep])
        if PARAM["peso_febrero"] > 1:
            w = w * np.where(pdf["periodo"].to_numpy() % 100 == 2, PARAM["peso_febrero"], 1.0)

    cl = pdf["cl"].to_numpy().astype(np.int16)
    return {"X": pdf[FEATS].to_numpy(dtype=np.float32), "y": y, "s": s, "w": w, "edad": edad,
            "prod": prod, "per": pdf["periodo"].to_numpy(), "cl": cl}


def peso(s_: np.ndarray, wb: np.ndarray, edad: np.ndarray, params: dict) -> np.ndarray:
    """`w = w_base * s^(p-1)`, con la MISMA p que la loss.

    Sin esto la hoja vale `Σ s^(1-p) y / Σ s^(2-p)`, o sea el ratio y/s ponderado por `s^(2-p)`.
    Con el factor, el peso efectivo queda en `s` y la hoja vale exactamente `Σy/Σs` — el ratio
    ponderado por tonelaje, que es lo que mide WAPE. Como Optuna mueve `p` entre 1,1 y 1,6, el
    peso TIENE que recalcularse por trial: con p fija en 1,3 el sesgo de la hoja va de -3,7% a
    +3,5% segun donde caiga Optuna (medido). `set_weight` no re-binnea, asi que sale gratis.
    """
    w = wb.copy()
    if PARAM["usar_peso_escala"] and PARAM["modo_escala"] == "offset":
        w = w * s_ ** (params["tweedie_variance_power"] - 1.0)
    vm = params.get("vida_media", 0) if PARAM["vida_media_rango"] else 0
    if vm and vm > 0:
        # peso por recencia: media vida `vm` meses. Tres anos de historia con todos los meses
        # pesando igual asume que el proceso no cambio, y en retail eso rara vez es cierto.
        w = w * 0.5 ** (edad / vm)
    return w


def etiqueta(y: np.ndarray, s_: np.ndarray) -> np.ndarray:
    """Clase del modo 'dividir': `tn(t+2)/s(t)`, con la convencion 0/0 := 0 del z702.

    Donde la escala es 0 (el par nunca vendio hasta t) el cociente no existe; forzarlo a 0 es lo
    que hacia el z702. Pierde esas filas como senal, pero evita que un y>0 sobre s~0 genere una
    etiqueta astronomica que el arbol despues persigue.
    """
    return np.where(s_ > PARAM["piso_escala"], y / np.maximum(s_, PARAM["piso_escala"]), 0.0)


def dataset(pack: dict, idx: np.ndarray) -> lgb.Dataset:
    dividir = PARAM["modo_escala"] == "dividir"
    lab = etiqueta(pack["y"][idx], pack["s"][idx]) if dividir else pack["y"][idx]
    ds = lgb.Dataset(pack["X"][idx], label=lab, weight=pack["w"][idx],
                     init_score=None if dividir else np.log(pack["s"][idx]),
                     feature_name=FEATS, categorical_feature=CAT_NOM,
                     free_raw_data=True,
                     params={"max_bin": PARAM["max_bin"], "verbosity": -1})
    ds.construct()
    return ds


def predecir(bst, X, s) -> np.ndarray:
    """Desescala la prediccion. El predict SIEMPRE vuelve a toneladas antes de agregarse."""
    if PARAM["modo_escala"] == "dividir":
        return np.maximum(bst.predict(X) * s, 0.0)
    f = np.clip(bst.predict(X, raw_score=True), -30, 30)
    return np.maximum(s * np.exp(f), 0.0)


def wape_producto(y, pred, prod) -> float:
    df = pd.DataFrame({"p": prod, "y": y, "h": pred}).groupby("p", sort=False).sum()
    den = df["y"].sum()
    return float(np.abs(df["y"] - df["h"]).sum() / den) if den > 0 else float("nan")


def grupos(prod: np.ndarray, per: np.ndarray, y: np.ndarray):
    """Indice compacto de ⟨producto, periodo⟩ y el tonelaje real agregado de cada grupo."""
    clave = prod.astype(np.int64) * 1000000 + per.astype(np.int64)
    _, inv = np.unique(clave, return_inverse=True)
    return inv.astype(np.int32), np.bincount(inv, weights=y)


def objetivo_agrupado(inv: np.ndarray, y_grupo: np.ndarray):
    """Gradiente = signo del error AGREGADO por ⟨producto, mes⟩. Es el numerador de WAPE.

    La pérdida por fila penaliza errores que la métrica perdona: si un par sobra 10 t y otro del
    mismo producto falta 10 t, Kaggle no cobra nada. Acá los errores se cancelan ANTES de
    penalizar, así que el modelo deja de gastar capacidad en ruido que no se mide.

    Requiere el agregado completo: con este objetivo se apagan el submuestreo de dormidos y el
    peso `s^(p−1)` (que existe para alinear la pérdida por fila, y acá ya no hace falta).
    """
    def fobj(preds, ds):
        mu = np.exp(np.clip(preds, -30, 30))
        agg = np.bincount(inv, weights=mu, minlength=len(y_grupo))
        sg = np.sign(agg - y_grupo)[inv]
        return mu * sg, mu + 1e-6
    return fobj


def fobj_de(g):
    """Objetivo custom agrupado (solo con modo offset); si no, None -> Tweedie nativo."""
    if PARAM["objetivo"] != "agrupado" or PARAM["modo_escala"] == "dividir" or g is None:
        return None
    return objetivo_agrupado(*g)


def entrenar(dtr, params, n, fobj=None):
    """Sin early stopping a proposito: el numero de arboles es un hiperparametro mas de Optuna.

    Parar por el fold de validacion lo usaria dos veces (para elegir n y para reportar el WAPE)
    y el numero que sale de ahi es optimista.
    """
    p = dict(params)
    if fobj is not None:
        p["objective"] = fobj
    return lgb.train(p, dtr, num_boost_round=n)


def params_base(trial=None, semilla=102191) -> dict:
    p = {"objective": "tweedie", "tweedie_variance_power": PARAM["tweedie_power"],
         "learning_rate": 0.03, "num_leaves": 127, "min_data_in_leaf": 300,
         "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
         "lambda_l1": 0.0, "lambda_l2": 0.0,
         "max_bin": PARAM["max_bin"], "num_threads": N_CORES, "verbosity": -1,
         "seed": semilla, "force_row_wise": True}
    if trial is not None:
        p.update({
            "tweedie_variance_power": trial.suggest_float("tvp", 1.1, 1.6),
            # rangos del z702, que son mas anchos que los que yo habia achicado
            "learning_rate": trial.suggest_float("lr", 5e-3, 0.2, log=True),
            "num_leaves": trial.suggest_int("leaves", 16, 256),
            "min_data_in_leaf": trial.suggest_int("mdl", 20, 300),
            "feature_fraction": trial.suggest_float("ff", 0.5, 1.0),
            "bagging_fraction": trial.suggest_float("bf", 0.6, 1.0),
            "lambda_l1": trial.suggest_float("l1", 1e-4, 10.0, log=True),
            "lambda_l2": trial.suggest_float("l2", 1e-4, 10.0, log=True),
        })
        if PARAM["vida_media_rango"]:
            p["vida_media"] = trial.suggest_float("vm", *PARAM["vida_media_rango"], log=True)
    else:
        p["vida_media"] = 0.0
    return p

## 7 — Folds

Un fold entrena hasta `ancla − 2` y evalúa en el ancla: el horizonte real del deploy. Los
`lgb.Dataset` se construyen **una vez por (fold, cluster)** y se reusan en todos los trials de
Optuna — con `max_bin=1023` el binning es lo caro, y la interfaz sklearn lo rehace en cada `fit`.

In [ ]:
FOLDS = []
for ancla in PARAM["anclas_optuna"]:
    corte = periodo_menos(ancla, PARAM["horizonte"])
    FOLDS.append({"ancla": ancla, "corte": corte, "mapa": CLUSTERS[corte],
                  "es_febrero": ancla % 100 == 2})
print("folds:", [(f["ancla"], f["corte"]) for f in FOLDS])

t0 = time.time()
for f in FOLDS:
    sem = PARAM["semillas_ensemble"][0]
    tr = paquete(f["corte"], "train", sem, f["mapa"])
    ev = paquete(f["corte"], "eval", sem, f["mapa"])
    f["ds"], f["grp"], f["s"], f["wb"], f["edad"] = {}, {}, {}, {}, {}
    for c in range(K):
        idx = np.where(tr["cl"] == c)[0]
        if len(idx) == 0:
            continue
        f["ds"][c] = dataset(tr, idx)
        f["grp"][c] = grupos(tr["prod"][idx], tr["per"][idx], tr["y"][idx])
        f["s"][c], f["wb"][c], f["edad"][c] = tr["s"][idx], tr["w"][idx], tr["edad"][idx]
    f["ev"] = ev
    f["n_tr"] = len(tr["y"])
    del tr; gc.collect()
    print(f"  ancla {f['ancla']}: train {f['n_tr']:,} filas, "
          f"eval {len(f['ev']['y']):,}, clusters {sorted(f['ds'])}")
print(f"datasets construidos en {time.time() - t0:.0f}s")

## 8 — Optuna, un estudio por cluster

Como cada producto vive en un solo cluster, `Σ_p |e_p|` se descompone exacto: optimizar el WAPE
**restringido a los productos del cluster** optimiza la métrica global. Eso es lo que habilita
entrenar y tunear por separado sin perder alineación con Kaggle.

Los folds de febrero pesan doble en el objetivo, porque el target es febrero.

In [ ]:
def evaluar_global(params: dict, n_arboles: int) -> float:
    """WAPE a nivel PRODUCTO sobre TODOS los clusters juntos.

    Con clusters por par, un producto se reparte entre varios clusters, asi que medir cluster por
    cluster seria sobre sumas parciales del producto: no es la metrica. Se entrena la config en
    todos los clusters, se agrega la prediccion completa y recien ahi se mide. Un solo estudio de
    Optuna en vez de K tambien tunea menos contra el ruido de cada fold.
    """
    num, den = 0.0, 0.0
    for f in FOLDS:
        ev = f["ev"]
        pred = np.zeros(len(ev["y"]))
        for c, ds in f["ds"].items():
            sel = ev["cl"] == c
            ds.set_weight(peso(f["s"][c], f["wb"][c], f["edad"][c], params))
            b = entrenar(ds, params, n_arboles, fobj_de(f["grp"].get(c)))
            if sel.sum():
                pred[sel] = predecir(b, ev["X"][sel], ev["s"][sel])
            del b; gc.collect()
        df = pd.DataFrame({"p": ev["prod"], "y": ev["y"], "h": pred}).groupby("p").sum()
        pe = PARAM["peso_febrero"] if f["es_febrero"] else 1.0
        num += pe * np.abs(df["y"] - df["h"]).sum()
        den += pe * df["y"].sum()
    return num / den if den > 0 else float("inf")


PISO_ARBOLES = max(40, min(150, PARAM["techo_arboles"] // 2))
CK_BEST = DIR_CK / "mejores_params.json"
if CK_BEST.exists():
    MEJOR = json.loads(CK_BEST.read_text())
    print("hiperparametros desde checkpoint")
else:
    t0 = time.time()
    if HAY_OPTUNA:
        st = optuna.create_study(direction="minimize", study_name="global",
                                 storage=f"sqlite:///{DIR_CK}/optuna.db", load_if_exists=True,
                                 sampler=optuna.samplers.TPESampler(seed=102191))

        def obj(trial):
            n = trial.suggest_int("arboles", PISO_ARBOLES, PARAM["techo_arboles"], log=True)
            return evaluar_global(params_base(trial), n)

        st.optimize(obj, n_trials=PARAM["n_trials"], show_progress_bar=False)
        bp = st.best_params
        MEJOR = {"params": {**params_base(), **{
            "tweedie_variance_power": bp["tvp"], "learning_rate": bp["lr"],
            "num_leaves": bp["leaves"], "min_data_in_leaf": bp["mdl"],
            "feature_fraction": bp["ff"], "bagging_fraction": bp["bf"],
            "lambda_l1": bp["l1"], "lambda_l2": bp["l2"],
            "vida_media": bp.get("vm", 0.0)}},
            "n_arboles": bp["arboles"], "wape_val": st.best_value}
    else:
        n = PARAM["techo_arboles"] // 2
        MEJOR = {"params": params_base(), "n_arboles": n,
                 "wape_val": evaluar_global(params_base(), n)}
    escribir_atomico(lambda f: Path(f).write_text(json.dumps(MEJOR, indent=2, default=str)), CK_BEST)
    print(f"  {PARAM['n_trials']} trials en {(time.time() - t0) / 60:.0f} min")

print(f"\nmejor WAPE de validacion (producto, global): {MEJOR['wape_val']:.4f}")
print(f"  p={MEJOR['params']['tweedie_variance_power']:.2f}  lr={MEJOR['params']['learning_rate']:.3f}  "
      f"hojas={MEJOR['params']['num_leaves']}  min_data={MEJOR['params']['min_data_in_leaf']}  "
      f"arboles={MEJOR['n_arboles']}  vida_media={MEJOR['params'].get('vida_media', 0):.0f} meses")

wape_global = float(MEJOR["wape_val"])

## 9 — Reporte mes a mes y multiplicador

Acá se ve si el modelo **generaliza a cualquier mes** o si sólo sabe hacer febrero. Se entrena con
los hiperparámetros elegidos y se mide en las siete anclas, una por una.

El ancla a mirar es **201812**: su origen es octubre, el segundo pico del año, y el target es
diciembre, el piso. Es el fold donde un modelo sin corrección estacional se estrella. Si ahí el
WAPE se dispara o el ratio `y/ŷ` cae muy por debajo de 1, `estacional_prod_objetivo` no está
haciendo su trabajo.

El multiplicador `m* = argmin_m Σ|y − m·ŷ|` es la **mediana ponderada** de `y_p/ŷ_p` con pesos
`ŷ_p`. Se estima en estos folds —nunca en el leaderboard— y se encoge hacia 1 con λ=0,7. Se
reporta el de cada mes: si varían mucho entre sí, un multiplicador global no es la herramienta.

In [ ]:
def multiplicador(y, yhat) -> float:
    ok = yhat > 0
    if ok.sum() == 0:
        return 1.0
    r, w = y[ok] / yhat[ok], yhat[ok]
    o = np.argsort(r); r, w = r[o], w[o]
    return float(r[np.searchsorted(np.cumsum(w), 0.5 * w.sum())])


def predecir_fold(corte: int, mapa, semillas=None) -> tuple:
    """Predice el fold con el MISMO ensemble que el modelo que se sube.

    Antes la validacion usaba una semilla y el modelo final tres: el WAPE reportado era mas
    ruidoso que el del modelo que efectivamente se manda, y el multiplicador salia de esa
    prediccion mas ruidosa. Sesgo pesimista, pero sesgo al fin.
    """
    semillas = semillas or PARAM["semillas_reporte"]
    tr = paquete(corte, "train", semillas[0], mapa)
    ev = paquete(corte, "eval", semillas[0], mapa)
    pred = np.zeros(len(ev["y"]))
    for c in range(K):
        itr, iev = np.where(tr["cl"] == c)[0], np.where(ev["cl"] == c)[0]
        if len(itr) == 0 or len(iev) == 0:
            continue
        d = dataset(tr, itr)
        d.set_weight(peso(tr["s"][itr], tr["w"][itr], tr["edad"][itr], MEJOR["params"]))
        fo = fobj_de(grupos(tr["prod"][itr], tr["per"][itr], tr["y"][itr]))
        acum = np.zeros(len(iev))
        for sem in semillas:
            b = entrenar(d, {**MEJOR["params"], "seed": sem}, MEJOR["n_arboles"], fo)
            acum += predecir(b, ev["X"][iev], ev["s"][iev])
            del b; gc.collect()
        pred[iev] = acum / len(semillas)
        del d; gc.collect()
    del tr; gc.collect()
    return ev, pred


REPORTE, GUARDADO = [], []
for ancla in PARAM["anclas_reporte"]:
    corte = periodo_menos(ancla, PARAM["horizonte"])
    t0 = time.time()
    ev, pred = predecir_fold(corte, CLUSTERS[corte])
    iv = FEATS.index("ventas_ult12") if "ventas_ult12" in FEATS else None
    im = FEATS.index("meses_desde_compra") if "meses_desde_compra" in FEATS else None
    GUARDADO.append({"ancla": ancla, "prod": ev["prod"], "y": ev["y"], "pred": pred,
                     "s": ev["s"],
                     "v12": ev["X"][:, iv] if iv is not None else np.ones(len(pred)),
                     "mdc": ev["X"][:, im] if im is not None else np.zeros(len(pred))})
    df = pd.DataFrame({"p": ev["prod"], "y": ev["y"], "h": pred}).groupby("p").sum()
    REPORTE.append({"ancla": ancla, "origen": corte,
                    "wape": float(np.abs(df["y"] - df["h"]).sum() / df["y"].sum()),
                    "m": multiplicador(df["y"].to_numpy(), df["h"].to_numpy()),
                    "real_tn": float(df["y"].sum()), "pred_tn": float(df["h"].sum())})
    print(f"  ancla {ancla} (origen {corte}): WAPE {REPORTE[-1]['wape']:.4f}  "
          f"m*={REPORTE[-1]['m']:.3f}  real {df['y'].sum():,.0f} vs pred {df['h'].sum():,.0f} tn  "
          f"({time.time() - t0:.0f}s)")
    del ev; gc.collect()

rep = pd.DataFrame(REPORTE)
rep["mes"] = rep["ancla"] % 100
print("\n" + rep[["ancla", "origen", "mes", "wape", "m", "real_tn", "pred_tn"]].round(4).to_string(index=False))
print(f"\nWAPE medio {rep['wape'].mean():.4f}  |  peor mes: ancla "
      f"{int(rep.loc[rep['wape'].idxmax(), 'ancla'])} con {rep['wape'].max():.4f}")
_oct = rep[rep.ancla == 201812]
if len(_oct):
    print(f"fold octubre->diciembre: WAPE {_oct['wape'].iloc[0]:.4f}, m*={_oct['m'].iloc[0]:.3f} "
          f"(m* muy por debajo de 1 = sigue sobre-prediciendo el pico de octubre)")

# --- A/B: ¿la estacionalidad del mes objetivo paga? ---
# Mismos hiperparametros, mismos folds, mismas semillas: lo unico que cambia son las 3 features.
# Se compara PAREADO por mes, que es mucho mas sensible que comparar promedios (cada mes tiene su
# propia dificultad y al restar se cancela).
SEASONALES = [f for f in ("mes_objetivo", "estacional_prod_objetivo", "avance_anual_prod")
              if f in FEATS]
if PARAM.get("ab_estacional", False) and SEASONALES:
    _base_f, _base_c = list(FEATS), list(CAT_NOM)
    FEATS = [f for f in _base_f if f not in SEASONALES]
    CAT_NOM = [c for c in _base_c if c in FEATS]
    print(f"\nA/B sin estacionalidad ({len(FEATS)} features, se sacan {SEASONALES}):")
    sin = []
    for ancla in PARAM["anclas_reporte"]:
        corte = periodo_menos(ancla, PARAM["horizonte"])
        ev, pred = predecir_fold(corte, CLUSTERS[corte])
        df = pd.DataFrame({"p": ev["prod"], "y": ev["y"], "h": pred}).groupby("p").sum()
        sin.append({"ancla": ancla,
                    "wape_sin": float(np.abs(df["y"] - df["h"]).sum() / df["y"].sum()),
                    "m_sin": multiplicador(df["y"].to_numpy(), df["h"].to_numpy())})
        print(f"  ancla {ancla}: WAPE {sin[-1]['wape_sin']:.4f}  m*={sin[-1]['m_sin']:.3f}")
        del ev; gc.collect()
    FEATS, CAT_NOM = _base_f, _base_c

    ab = rep.merge(pd.DataFrame(sin), on="ancla")
    ab["delta"] = ab["wape"] - ab["wape_sin"]
    print("\n" + ab[["ancla", "wape_sin", "wape", "delta", "m_sin", "m"]].round(4).to_string(index=False))
    d, ee = ab["delta"].mean(), ab["delta"].std(ddof=1) / np.sqrt(len(ab))
    print(f"\ndelta medio {d:+.4f} +- {ee:.4f} (error est.)  |  mejora en {(ab.delta < 0).sum()}/{len(ab)} meses")
    print("veredicto:", "LA ESTACIONALIDAD PAGA" if d < -2 * ee else
          ("EMPEORA" if d > 2 * ee else "DENTRO DEL RUIDO: no hay evidencia de que sirva"))
else:
    ab = None

# El multiplicador sale SOLO de las anclas que Optuna no vio: usar las mismas que se tunearon
# da un m* optimista, porque los hiperparametros ya se eligieron para que ese fold quede bien.
LIMPIAS = [a for a in PARAM["anclas_reporte"] if a not in PARAM["anclas_optuna"]]
lim = rep[rep.ancla.isin(LIMPIAS)]
if lim.empty:
    lim = rep
    print("AVISO: no quedan anclas limpias, el multiplicador sale de folds ya tuneados")
print(f"multiplicador estimado sobre anclas limpias {LIMPIAS} "
      f"({(lim['mes'] == 2).sum()} de febrero)")
pesos = np.where(lim["mes"] == 2, PARAM["peso_febrero"], 1.0)
m_bar = float(np.average(lim["m"], weights=pesos))
M_FINAL = 1 + PARAM["lambda_shrink"] * (m_bar - 1)
print(f"\nm̄ (ponderando febrero x{PARAM['peso_febrero']}) = {m_bar:.3f}  ->  "
      f"m_final (shrink {PARAM['lambda_shrink']}) = {M_FINAL:.3f}")
print(f"dispersion de m* entre meses: {lim['m'].std():.3f} "
      f"({'homogeneo, un multiplicador sirve' if lim['m'].std() < 0.08 else 'ALTO: el sesgo cambia con el mes, desconfia del multiplicador global'})")

## 10 — Reglas de post-proceso (baratas, y sólo si ganan)

Cuatro reglas de una línea, evaluadas sobre las predicciones de los folds **ya calculadas**: no
hay que reentrenar nada. Dos apagan pares muertos —el modelo les da un positivo chiquito y, con
cientos de miles de pares, eso suma tonelaje fantasma— y dos recortan predicciones absurdas
respecto de la propia escala del par.

Se aplica **una sola**, y sólo si le gana a no hacer nada por más de `margen_regla` en promedio
**y** en la mayoría de los folds. La lección del multiplicador está fresca: una mejora promedio
chica que no es consistente entre folds es ruido, y aplicarla costó caro siete veces.

In [ ]:
def aplicar(g: dict, regla: str) -> np.ndarray:
    h = g["pred"].copy()
    if regla == "cero_si_dormido_12":
        h[g["v12"] == 0] = 0.0
    elif regla == "cero_si_sin_compra_6":
        h[g["mdc"] >= 6] = 0.0
    elif regla.startswith("clip_"):
        k = float(regla.split("_")[1])
        h = np.minimum(h, k * g["s"])
    return h


REGLAS = ["base", "cero_si_dormido_12", "cero_si_sin_compra_6", "clip_3", "clip_5", "clip_10"]
tabla = []
for r in REGLAS:
    fila = {"regla": r}
    for g in GUARDADO:
        h = g["pred"] if r == "base" else aplicar(g, r)
        df = pd.DataFrame({"p": g["prod"], "y": g["y"], "h": h}).groupby("p").sum()
        fila[g["ancla"]] = float(np.abs(df["y"] - df["h"]).sum() / df["y"].sum())
    tabla.append(fila)
tab_r = pd.DataFrame(tabla).set_index("regla")
tab_r["promedio"] = tab_r.mean(axis=1)
print(tab_r.round(4).to_string())

base_f = tab_r.loc["base", PARAM["anclas_reporte"]]
mejor_r, delta_r, gana_r = "base", 0.0, 0
for r in REGLAS[1:]:
    d = (tab_r.loc[r, PARAM["anclas_reporte"]] - base_f)
    if d.mean() < delta_r and (d < 0).sum() > len(d) / 2:
        mejor_r, delta_r, gana_r = r, d.mean(), int((d < 0).sum())
if mejor_r != "base" and abs(delta_r) >= PARAM["margen_regla"]:
    print(f"\nse aplica '{mejor_r}': {delta_r:+.4f} de WAPE, gana en {gana_r}/{len(GUARDADO)} folds")
    REGLA = mejor_r
else:
    print(f"\nninguna regla supera el margen de {PARAM['margen_regla']:.3f} de forma consistente "
          f"-> se predice en crudo")
    REGLA = "base"

## 10 — Entrenamiento final e inferencia

Se entrena con **toda** la historia hasta 201912 y se predice 202002. Ensemble de 3 semillas por
cluster, promediado. Los clusters de inferencia son los calculados con corte 201912.

In [ ]:
mapa_inf = CLUSTERS[PARAM["periodo_inferencia"]]
tr = paquete(PARAM["periodo_inferencia"], "train", PARAM["semillas_ensemble"][0], mapa_inf)
inf = paquete(PARAM["periodo_inferencia"], "eval", PARAM["semillas_ensemble"][0], mapa_inf)
print(f"train final: {len(tr['y']):,} filas | inferencia: {len(inf['y']):,} filas")

pred = np.zeros(len(inf["y"]))
for c in range(K):
    itr, iev = np.where(tr["cl"] == c)[0], np.where(inf["cl"] == c)[0]
    if len(itr) == 0 or len(iev) == 0:
        continue
    t0 = time.time()
    d = dataset(tr, itr)
    d.set_weight(peso(tr["s"][itr], tr["w"][itr], tr["edad"][itr], MEJOR["params"]))
    fobj_fin = fobj_de(grupos(tr["prod"][itr], tr["per"][itr], tr["y"][itr]))
    acum = np.zeros(len(iev))
    for sem in PARAM["semillas_ensemble"]:
        b = entrenar(d, {**MEJOR["params"], "seed": sem}, MEJOR["n_arboles"], fobj_fin)
        acum += predecir(b, inf["X"][iev], inf["s"][iev])
        del b; gc.collect()
    pred[iev] = acum / len(PARAM["semillas_ensemble"])
    del d; gc.collect()
    print(f"  cluster {c}: {len(itr):,} filas train, {len(iev):,} de inferencia "
          f"({time.time() - t0:.0f}s)")

del tr; gc.collect()

if REGLA != "base":
    iv = FEATS.index("ventas_ult12") if "ventas_ult12" in FEATS else None
    im = FEATS.index("meses_desde_compra") if "meses_desde_compra" in FEATS else None
    antes = pred.sum()
    pred = aplicar({"pred": pred, "s": inf["s"],
                    "v12": inf["X"][:, iv] if iv is not None else np.ones(len(pred)),
                    "mdc": inf["X"][:, im] if im is not None else np.zeros(len(pred))}, REGLA)
    print(f"regla '{REGLA}' aplicada a la inferencia: {antes:,.0f} -> {pred.sum():,.0f} tn")

sub = (pd.DataFrame({"product_id": inf["prod"], "tn": pred})
       .groupby("product_id", as_index=False).sum())
sub = sub[sub["product_id"].isin(PROD_TARGET)].sort_values("product_id").reset_index(drop=True)
faltan = set(PROD_TARGET) - set(sub["product_id"])
if faltan:
    sub = pd.concat([sub, pd.DataFrame({"product_id": sorted(faltan), "tn": 0.0})]
                    ).sort_values("product_id").reset_index(drop=True)
print(f"\nsubmit: {len(sub)} productos ({len(faltan)} sin prediccion, van en 0)")

## 11 — Test de nivel y submits

La banda **28 400 – 30 600 tn** sale de los febreros reales de los 780 (27 304 / 28 185 / 27 200
en 2017-19) corrigiendo por los 120 productos que no existían en feb-2019 y que hoy aportan
1 574 tn/mes. Es un chequeo de cordura, no un juez: si el total cae afuera, hay que mirar por qué
antes de subir. El z705 dio 26 588 y se subió igual.

In [ ]:
lo, hi = PARAM["banda_nivel"]
for nom, v in [("m=1.000", sub["tn"].values), (f"m={M_FINAL:.3f}", sub["tn"].values * M_FINAL)]:
    t = v.sum()
    estado = "DENTRO" if lo <= t <= hi else ("ALTO" if t > hi else "BAJO")
    print(f"  {nom}: total {t:,.0f} tn -> {estado} de la banda [{lo:,}, {hi:,}]")

WAPE_REPORTE = float(rep["wape"].mean())
MODO = PARAM["modo_escala"]
CANDIDATOS = ([("crudo", 1.0), ("mfinal", M_FINAL)] if PARAM["aplicar_multiplicador"]
              else [("crudo", 1.0)])
print(f"m* diagnostico = {m_bar:.3f} (m_final seria {M_FINAL:.3f}), "
      f"{'se aplica' if PARAM['aplicar_multiplicador'] else 'NO se aplica: 7/7 veces perdio en el LB'}")
SUBMITS = []
for nom, mult in CANDIDATOS:
    d = sub.copy(); d["tn"] = np.maximum(d["tn"] * mult, 0.0)
    f = DIR_OUT / f"z711_{nom}_m{mult:.3f}.csv"
    escribir_atomico(lambda p, d=d: d.to_csv(p, index=False), f)
    SUBMITS.append((f, f"z711 = z702 corregido: {MODO}, DTW por par, {len(FEATS)} feats, "
                       f"K={K}, WAPE val {wape_global:.4f} / reporte {WAPE_REPORTE:.4f}, m={mult:.3f}"))
    print(f"  {f.name}: {d['tn'].sum():,.0f} tn")

(DIR_EXP / "resumen.json").write_text(json.dumps({
    "mejor": MEJOR,
    "wape_val": wape_global, "reporte_por_mes": REPORTE,
    "ab_estacional": (ab.to_dict("records") if ab is not None else None),
    "m_bar": m_bar, "m_final": M_FINAL, "K": K,
    "features": FEATS, "regla": REGLA, "total_tn": float(sub["tn"].sum() * M_FINAL),
    "param": {k: v for k, v in PARAM.items() if k != "features"}}, indent=2, default=str))


def kaggle_submit(comp, archivo: Path, msg: str):
    flag = archivo.with_suffix(".done")
    if flag.exists():
        print("ya subido:", archivo.name); return
    r = subprocess.run(["kaggle", "competitions", "submit", "-c", comp,
                        "-f", str(archivo), "-m", msg], capture_output=True, text=True)
    print(archivo.name, "->", (r.stdout or r.stderr).strip()[:180])
    if r.returncode == 0:
        flag.write_text(msg)


if PARAM["submit"]:
    for f, msg in SUBMITS:
        kaggle_submit(PARAM["kaggle_competition"], f, msg)
else:
    print("submit=False -> CSVs en", DIR_OUT)